[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/16_planificacion_clasica/notebooks/02_planificacion_forward.ipynb)

# Notebook 02 — Planificación forward y backward

En el notebook anterior definimos la representación STRIPS: estados como conjuntos de
proposiciones y acciones con precondiciones, lista add y lista delete.

Ahora conectamos esa representación con **búsqueda genérica** (módulo 13). La idea
central es que la planificación hacia adelante (*forward planning*) **es**
`GENERIC-SEARCH` con tres sustituciones:

| Componente | Búsqueda genérica | Planificación forward |
|---|---|---|
| Test de meta | `es_meta(n)` compara contra un estado | `meta ⊆ s` — la meta es un subconjunto |
| Vecinos | `vecinos(n)` — aristas predefinidas | `aplicables(s)` — acciones con pre ⊆ s |
| Transición | seguir la arista | `aplicar(s, a)` — s − delete ∪ add |

En este notebook:

1. Redefinimos la infraestructura STRIPS (para que el notebook sea autocontenido)
2. Implementamos BFS forward con traza detallada
3. Implementamos la heurística de *delete relaxation*
4. Implementamos A* forward con esa heurística
5. Comparamos BFS vs A* en problemas de 3, 4 y 5 bloques
6. Implementamos búsqueda **hacia atrás** (regresión) y comparamos con forward

> **Prerequisito:** haber leído `02_strips.md`, `03_busqueda_hacia_adelante.md` y `05_busqueda_hacia_atras.md` en las notas del curso.

---
## Sección 1: Infraestructura STRIPS

Redefinimos aquí toda la maquinaria STRIPS para que el notebook sea completamente
autocontenido. Esto incluye:

- La representación de acciones como `namedtuple`
- `is_applicable` — verificar precondiciones
- `apply_action` — calcular el estado sucesor
- `make_blocks_world_actions` — generar las 18 acciones concretas para 3 bloques

Los estados son `frozenset` de cadenas (proposiciones). Las acciones usan `frozenset`
para precondiciones, lista add y lista delete.

In [1]:
from collections import namedtuple, deque
import heapq
import time

# ── STRIPS action representation ─────────────────────────────────────────────
# Each action has: name, preconditions, add_list, delete_list
# All sets are frozenset (immutable) so states can be used as dict keys.

Action = namedtuple("Action", ["name", "preconditions", "add_list", "delete_list"])


def is_applicable(state, action):
    """
    Check if an action is applicable in a given state.

    An action is applicable iff ALL its preconditions are in the state.

    Args:
        state  : frozenset — set of true propositions
        action : Action   — STRIPS action

    Returns:
        bool — True if preconditions ⊆ state
    """
    return action.preconditions.issubset(state)


def apply_action(state, action):
    """
    Apply an action to a state and return the successor state.

    Formula: state' = (state − delete_list) ∪ add_list

    Args:
        state  : frozenset — current state
        action : Action   — action to apply (must be applicable)

    Returns:
        frozenset — new state after applying the action
    """
    return (state - action.delete_list) | action.add_list


def get_applicable(state, all_actions):
    """
    Return all actions whose preconditions are satisfied in state.

    Args:
        state       : frozenset — current state
        all_actions : list[Action] — all domain actions

    Returns:
        list[Action] — applicable actions
    """
    return [a for a in all_actions if is_applicable(state, a)]


print("Infraestructura STRIPS definida: Action, is_applicable, apply_action, get_applicable")

Infraestructura STRIPS definida: Action, is_applicable, apply_action, get_applicable


### Generador de acciones para Blocks World

La función `make_blocks_world_actions` genera **todas** las acciones concretas
(ground actions) para un conjunto de bloques dado. Los tres esquemas son:

- **Mover(X, Y, Z)**: mover X de bloque Y a bloque Z
- **MoverAMesa(X, Y)**: mover X de bloque Y a la mesa
- **MoverDesdeMesa(X, Z)**: mover X de la mesa a bloque Z

Para $n$ bloques se generan $n(n-1)(n-2) + 2 \cdot n(n-1)$ acciones.

In [2]:
def make_blocks_world_actions(blocks):
    """
    Generate all ground STRIPS actions for Blocks World.

    Three action schemas:
      Mover(X, Y, Z)       — move X from block Y to block Z
      MoverAMesa(X, Y)     — move X from block Y to the table
      MoverDesdeMesa(X, Z) — move X from the table to block Z

    Args:
        blocks : list[str] — list of block names, e.g. ["A", "B", "C"]

    Returns:
        list[Action] — all ground actions
    """
    actions = []

    for x in blocks:
        for y in blocks:
            if x == y:
                continue
            for z in blocks:
                if z == x or z == y:
                    continue
                # Mover(X, Y, Z): move X from block Y to block Z
                actions.append(Action(
                    name=f"Mover({x},{y},{z})",
                    preconditions=frozenset({f"On({x},{y})", f"Clear({x})", f"Clear({z})"}),
                    add_list=frozenset({f"On({x},{z})", f"Clear({y})"}),
                    delete_list=frozenset({f"On({x},{y})", f"Clear({z})"}),
                ))

        for y in blocks:
            if x == y:
                continue
            # MoverAMesa(X, Y): move X from block Y to table
            actions.append(Action(
                name=f"MoverAMesa({x},{y})",
                preconditions=frozenset({f"On({x},{y})", f"Clear({x})"}),
                add_list=frozenset({f"On({x},Mesa)", f"Clear({y})"}),
                delete_list=frozenset({f"On({x},{y})"}),
            ))

            # MoverDesdeMesa(X, Y): move X from table to block Y
            actions.append(Action(
                name=f"MoverDesdeMesa({x},{y})",
                preconditions=frozenset({f"On({x},Mesa)", f"Clear({x})", f"Clear({y})"}),
                add_list=frozenset({f"On({x},{y})"}),
                delete_list=frozenset({f"On({x},Mesa)", f"Clear({y})"}),
            ))

    return actions


# ── Quick test with 3 blocks ──────────────────────────────────────────────────
blocks_3 = ["A", "B", "C"]
actions_3 = make_blocks_world_actions(blocks_3)
print(f"Bloques: {blocks_3}")
print(f"Total de acciones generadas: {len(actions_3)}")
print(f"  Mover:          {sum(1 for a in actions_3 if a.name.startswith('Mover('))}")
print(f"  MoverAMesa:     {sum(1 for a in actions_3 if a.name.startswith('MoverAMesa'))}")
print(f"  MoverDesdeMesa: {sum(1 for a in actions_3 if a.name.startswith('MoverDesdeMesa'))}")

Bloques: ['A', 'B', 'C']
Total de acciones generadas: 18
  Mover:          6
  MoverAMesa:     6
  MoverDesdeMesa: 6


---
## Sección 2: BFS Forward Planner con traza

Implementamos `FORWARD-PLANNING` usando BFS (cola FIFO). La frontera es un `deque`.
En cada iteración del bucle principal imprimimos:

1. El estado actual sacado de la frontera
2. Cuántas acciones son aplicables
3. Los estados nuevos generados
4. El tamaño de la frontera

Esto permite seguir la exploración paso a paso.

In [3]:
def format_state(state):
    """
    Pretty-print a STRIPS state as a sorted list of propositions.

    Args:
        state : frozenset — set of propositions

    Returns:
        str — formatted string
    """
    return "{" + ", ".join(sorted(state)) + "}"


def forward_bfs(initial, goal, all_actions, verbose=False):
    """
    Forward planning with BFS (FIFO frontier).

    Args:
        initial     : frozenset — initial state
        goal        : frozenset — goal propositions (must be subset of final state)
        all_actions : list[Action] — all ground actions
        verbose     : bool — if True, print detailed trace

    Returns:
        tuple(list[Action], int) — (plan, nodes_explored)
        or (None, nodes_explored) if no plan exists
    """
    # ── Initialization ────────────────────────────────────────────────────
    frontier = deque([initial])          # FIFO queue for BFS
    explored = set()                     # states already processed
    parent = {initial: (None, None)}     # state -> (parent_state, action_used)
    nodes_explored = 0                   # counter for comparison

    # ── Main loop ─────────────────────────────────────────────────────────
    while frontier:
        s = frontier.popleft()           # BFS: pop oldest state
        nodes_explored += 1

        if verbose:
            print(f"\n--- Iteración {nodes_explored} ---")
            print(f"  Estado actual:      {format_state(s)}")

        # ── Goal test [D1] ────────────────────────────────────────────────
        if goal.issubset(s):
            if verbose:
                print(f"  ** META ALCANZADA **")
            # Reconstruct plan by following parent pointers
            plan = []
            current = s
            while parent[current][0] is not None:
                prev_state, action = parent[current]
                plan.append(action)
                current = prev_state
            plan.reverse()
            return plan, nodes_explored

        explored.add(s)

        # ── Expand [D2] and [D3] ──────────────────────────────────────────
        applicable = get_applicable(s, all_actions)
        if verbose:
            print(f"  Acciones aplicables: {len(applicable)}")

        new_count = 0
        for a in applicable:
            new_state = apply_action(s, a)        # [D3] transition
            if new_state not in explored and new_state not in set(frontier):
                parent[new_state] = (s, a)
                frontier.append(new_state)
                new_count += 1
                if verbose:
                    print(f"    + {a.name} -> {format_state(new_state)}")

        if verbose:
            print(f"  Nuevos estados agregados: {new_count}")
            print(f"  Tamaño de frontera: {len(frontier)}")

    return None, nodes_explored            # no plan found


print("forward_bfs definida.")

forward_bfs definida.


### Resolver el problema de 3 bloques

Estado inicial: A, B, C todos en la mesa.
Meta: A sobre B, B sobre C, C en la mesa (torre A/B/C).

Ejecutamos BFS con `verbose=True` para ver la traza completa de exploración.

In [4]:
# ── 3-block problem definition ───────────────────────────────────────────────
initial_3 = frozenset({
    "On(A,Mesa)", "On(B,Mesa)", "On(C,Mesa)",
    "Clear(A)", "Clear(B)", "Clear(C)"
})

goal_3 = frozenset({
    "On(A,B)", "On(B,C)", "On(C,Mesa)", "Clear(A)"
})

print("Estado inicial:", format_state(initial_3))
print("Meta:          ", format_state(goal_3))
print()

# ── Solve with verbose trace ──────────────────────────────────────────────────
plan_3, explored_3 = forward_bfs(initial_3, goal_3, actions_3, verbose=True)

Estado inicial: {Clear(A), Clear(B), Clear(C), On(A,Mesa), On(B,Mesa), On(C,Mesa)}
Meta:           {Clear(A), On(A,B), On(B,C), On(C,Mesa)}


--- Iteración 1 ---
  Estado actual:      {Clear(A), Clear(B), Clear(C), On(A,Mesa), On(B,Mesa), On(C,Mesa)}
  Acciones aplicables: 6
    + MoverDesdeMesa(A,B) -> {Clear(A), Clear(C), On(A,B), On(B,Mesa), On(C,Mesa)}
    + MoverDesdeMesa(A,C) -> {Clear(A), Clear(B), On(A,C), On(B,Mesa), On(C,Mesa)}
    + MoverDesdeMesa(B,A) -> {Clear(B), Clear(C), On(A,Mesa), On(B,A), On(C,Mesa)}
    + MoverDesdeMesa(B,C) -> {Clear(A), Clear(B), On(A,Mesa), On(B,C), On(C,Mesa)}
    + MoverDesdeMesa(C,A) -> {Clear(B), Clear(C), On(A,Mesa), On(B,Mesa), On(C,A)}
    + MoverDesdeMesa(C,B) -> {Clear(A), Clear(C), On(A,Mesa), On(B,Mesa), On(C,B)}
  Nuevos estados agregados: 6
  Tamaño de frontera: 6

--- Iteración 2 ---
  Estado actual:      {Clear(A), Clear(C), On(A,B), On(B,Mesa), On(C,Mesa)}
  Acciones aplicables: 3
    + MoverDesdeMesa(C,A) -> {Clear(C), On(A,B), O

### Resumen del plan encontrado

Mostramos el plan paso a paso y comparamos nodos explorados vs longitud del plan.

In [5]:
print("=" * 60)
print("RESUMEN — BFS Forward Planning (3 bloques)")
print("=" * 60)
print(f"Plan encontrado: {len(plan_3)} pasos")
print(f"Nodos explorados: {explored_3}")
print(f"Ratio explorados/plan: {explored_3 / len(plan_3):.1f}x")
print()

# Show plan step by step
print("Plan:")
state = initial_3
for i, action in enumerate(plan_3, 1):
    print(f"  Paso {i}: {action.name}")
    state = apply_action(state, action)
    print(f"          Estado resultante: {format_state(state)}")

# Verify goal is reached
assert goal_3.issubset(state), "ERROR: la meta no se alcanzó"
print(f"\nMeta alcanzada: {goal_3.issubset(state)}")

RESUMEN — BFS Forward Planning (3 bloques)
Plan encontrado: 2 pasos
Nodos explorados: 11
Ratio explorados/plan: 5.5x

Plan:
  Paso 1: MoverDesdeMesa(B,C)
          Estado resultante: {Clear(A), Clear(B), On(A,Mesa), On(B,C), On(C,Mesa)}
  Paso 2: MoverDesdeMesa(A,B)
          Estado resultante: {Clear(A), On(A,B), On(B,C), On(C,Mesa)}

Meta alcanzada: True


---
## Sección 3: Heurística de *delete relaxation*

BFS explora muchos nodos porque no tiene información sobre cuán "lejos" está un
estado de la meta. Para guiar la búsqueda, necesitamos una **heurística** $h(s)$ que
estime el costo restante.

La idea de *delete relaxation* es simple y elegante:

> **Relajar el problema**: resolver el mismo problema pero **ignorando las listas delete**
> de todas las acciones. Es decir, las acciones solo agregan proposiciones, nunca quitan.

Esto hace el problema más fácil (nunca perdemos progreso), así que la longitud del plan
relajado es una **subestimación** del plan real: $h^+(s) \leq h^*(s)$.
Por lo tanto, $h^+$ es una heurística **admisible** y podemos usarla con A*.

La implementación usa BFS en el problema relajado (sin delete lists) y devuelve la
longitud del plan encontrado como estimación.

In [6]:
def h_delete_relaxation(state, goal, all_actions):
    """
    Delete-relaxation heuristic.

    Solves a relaxed version of the problem where all delete lists are empty.
    Returns the length of the relaxed plan as the heuristic estimate.

    This is admissible: h+(s) <= h*(s), because the relaxed problem is
    strictly easier (actions never undo progress).

    Args:
        state       : frozenset — current state
        goal        : frozenset — goal propositions
        all_actions : list[Action] — all ground actions

    Returns:
        int — estimated cost (relaxed plan length), or float('inf') if unsolvable
    """
    # If goal already satisfied, cost is 0
    if goal.issubset(state):
        return 0

    # Create relaxed actions: same as original but delete_list = empty
    relaxed_actions = [
        Action(
            name=a.name,
            preconditions=a.preconditions,
            add_list=a.add_list,
            delete_list=frozenset()    # ignore deletes
        )
        for a in all_actions
    ]

    # BFS on the relaxed problem
    frontier = deque([(state, 0)])       # (state, depth)
    explored = set()
    explored.add(state)

    while frontier:
        s, depth = frontier.popleft()

        # Try all relaxed actions
        for a in relaxed_actions:
            if a.preconditions.issubset(s):
                new_state = s | a.add_list    # only add, never delete

                # Check goal in the relaxed successor
                if goal.issubset(new_state):
                    return depth + 1

                if new_state not in explored:
                    explored.add(new_state)
                    frontier.append((new_state, depth + 1))

    return float('inf')   # relaxed problem unsolvable => original also unsolvable


# ── Test the heuristic ─────────────────────────────────────────────────────────
h_init = h_delete_relaxation(initial_3, goal_3, actions_3)
print(f"h+(estado_inicial) = {h_init}")
print(f"Plan real tiene {len(plan_3)} pasos")
print(f"h+ <= h* ? {h_init <= len(plan_3)} (admisible)")

h+(estado_inicial) = 2
Plan real tiene 2 pasos
h+ <= h* ? True (admisible)


---
## Sección 4: A* Forward Planner

Ahora combinamos la búsqueda forward con la heurística de delete relaxation usando
**A***. La frontera es una cola de prioridad ordenada por:

$$f(s) = g(s) + h^+(s)$$

donde:
- $g(s)$ = número de acciones desde el estado inicial hasta $s$
- $h^+(s)$ = heurística de delete relaxation (estimación del costo restante)

Como $h^+$ es admisible, A* garantiza encontrar un plan óptimo.

In [7]:
def forward_astar(initial, goal, all_actions, verbose=False):
    """
    Forward planning with A* search.

    Uses delete-relaxation heuristic for h(s).
    Priority queue ordered by f(s) = g(s) + h(s).

    Args:
        initial     : frozenset — initial state
        goal        : frozenset — goal propositions
        all_actions : list[Action] — all ground actions
        verbose     : bool — if True, print trace

    Returns:
        tuple(list[Action], int) — (plan, nodes_explored)
        or (None, nodes_explored) if no plan exists
    """
    # ── Initialization ────────────────────────────────────────────────────
    h_init = h_delete_relaxation(initial, goal, all_actions)
    counter = 0                          # tiebreaker for heap
    # Heap entries: (f, counter, state, g)
    frontier = [(h_init, counter, initial, 0)]
    best_g = {initial: 0}                # state -> best g known
    parent = {initial: (None, None)}     # state -> (parent_state, action)
    explored = set()                     # closed set
    nodes_explored = 0

    # ── Main loop ─────────────────────────────────────────────────────────
    while frontier:
        f, _, s, g = heapq.heappop(frontier)

        # Skip if we already found a better path to this state
        if s in explored:
            continue

        nodes_explored += 1

        if verbose:
            print(f"\n--- Iteración {nodes_explored} ---")
            print(f"  Estado: {format_state(s)}")
            print(f"  g={g}, f={f}")

        # ── Goal test ─────────────────────────────────────────────────────
        if goal.issubset(s):
            if verbose:
                print(f"  ** META ALCANZADA **")
            # Reconstruct plan
            plan = []
            current = s
            while parent[current][0] is not None:
                prev_state, action = parent[current]
                plan.append(action)
                current = prev_state
            plan.reverse()
            return plan, nodes_explored

        explored.add(s)

        # ── Expand ────────────────────────────────────────────────────────
        applicable = get_applicable(s, all_actions)
        if verbose:
            print(f"  Acciones aplicables: {len(applicable)}")

        for a in applicable:
            new_state = apply_action(s, a)
            new_g = g + 1                # uniform action cost

            if new_state in explored:
                continue

            # Only add if this is a better path
            if new_state not in best_g or new_g < best_g[new_state]:
                best_g[new_state] = new_g
                parent[new_state] = (s, a)
                h = h_delete_relaxation(new_state, goal, all_actions)
                f_new = new_g + h
                counter += 1
                heapq.heappush(frontier, (f_new, counter, new_state, new_g))

                if verbose:
                    print(f"    + {a.name}  g={new_g}, h={h}, f={f_new}")

    return None, nodes_explored


print("forward_astar definida.")

forward_astar definida.


### A* en el problema de 3 bloques

Resolvemos el mismo problema de 3 bloques con A* y traza para ver cómo la heurística
guía la búsqueda hacia estados más prometedores.

In [8]:
plan_astar_3, explored_astar_3 = forward_astar(initial_3, goal_3, actions_3, verbose=True)

print("\n" + "=" * 60)
print("RESUMEN — A* Forward Planning (3 bloques)")
print("=" * 60)
print(f"Plan encontrado: {len(plan_astar_3)} pasos")
print(f"Nodos explorados: {explored_astar_3}")
print()
print("Plan:")
for i, action in enumerate(plan_astar_3, 1):
    print(f"  Paso {i}: {action.name}")


--- Iteración 1 ---
  Estado: {Clear(A), Clear(B), Clear(C), On(A,Mesa), On(B,Mesa), On(C,Mesa)}
  g=0, f=2
  Acciones aplicables: 6
    + MoverDesdeMesa(A,B)  g=1, h=2, f=3
    + MoverDesdeMesa(A,C)  g=1, h=2, f=3
    + MoverDesdeMesa(B,A)  g=1, h=2, f=3
    + MoverDesdeMesa(B,C)  g=1, h=1, f=2
    + MoverDesdeMesa(C,A)  g=1, h=3, f=4
    + MoverDesdeMesa(C,B)  g=1, h=3, f=4

--- Iteración 2 ---
  Estado: {Clear(A), Clear(B), On(A,Mesa), On(B,C), On(C,Mesa)}
  g=1, f=2
  Acciones aplicables: 3
    + MoverDesdeMesa(A,B)  g=2, h=0, f=2

--- Iteración 3 ---
  Estado: {Clear(A), On(A,B), On(B,C), On(C,Mesa)}
  g=2, f=2
  ** META ALCANZADA **

RESUMEN — A* Forward Planning (3 bloques)
Plan encontrado: 2 pasos
Nodos explorados: 3

Plan:
  Paso 1: MoverDesdeMesa(B,C)
  Paso 2: MoverDesdeMesa(A,B)


---
## Sección 5: Comparación BFS vs A* (3 bloques)

Comparamos ambos algoritmos en el problema de 3 bloques midiendo:
- Nodos explorados (expandidos)
- Longitud del plan encontrado
- Tiempo de ejecución

Para 3 bloques la diferencia puede ser pequeña (el espacio tiene solo 13 estados),
pero la tendencia se amplifica con problemas más grandes.

In [9]:
def benchmark(name, search_fn, initial, goal, all_actions):
    """
    Run a search function and report results with timing.

    Args:
        name       : str — algorithm name for display
        search_fn  : callable — forward_bfs or forward_astar
        initial    : frozenset — initial state
        goal       : frozenset — goal propositions
        all_actions: list[Action] — all ground actions

    Returns:
        dict — results with plan_length, nodes_explored, time_ms
    """
    t0 = time.time()
    plan, nodes = search_fn(initial, goal, all_actions, verbose=False)
    t1 = time.time()
    elapsed_ms = (t1 - t0) * 1000

    result = {
        "name": name,
        "plan_length": len(plan) if plan else None,
        "nodes_explored": nodes,
        "time_ms": elapsed_ms,
        "plan": plan,
    }
    return result


def print_comparison(results):
    """
    Print a comparison table of benchmark results.

    Args:
        results : list[dict] — list of benchmark results
    """
    print(f"{'Algoritmo':<15} {'Plan':>6} {'Nodos':>8} {'Tiempo (ms)':>12}")
    print("-" * 45)
    for r in results:
        plan_str = str(r['plan_length']) if r['plan_length'] is not None else "N/A"
        print(f"{r['name']:<15} {plan_str:>6} {r['nodes_explored']:>8} {r['time_ms']:>12.2f}")


# ── Compare on 3-block problem ─────────────────────────────────────────────────
print("=== Comparación: 3 bloques (A,B,C en mesa -> torre A/B/C) ===\n")

r_bfs_3 = benchmark("BFS", forward_bfs, initial_3, goal_3, actions_3)
r_astar_3 = benchmark("A*", forward_astar, initial_3, goal_3, actions_3)

print_comparison([r_bfs_3, r_astar_3])

print(f"\nAmbos encuentran plan de {r_bfs_3['plan_length']} pasos.")
print(f"A* explora {r_bfs_3['nodes_explored'] - r_astar_3['nodes_explored']} nodos menos que BFS.")

=== Comparación: 3 bloques (A,B,C en mesa -> torre A/B/C) ===

Algoritmo         Plan    Nodos  Tiempo (ms)
---------------------------------------------
BFS                  2       11         0.26
A*                   2        3         0.56

Ambos encuentran plan de 2 pasos.
A* explora 8 nodos menos que BFS.


---
## Sección 6: Escalando — 4 y 5 bloques

El verdadero valor de las heurísticas se ve cuando el espacio de estados crece.
Para $n$ bloques, el número de estados posibles crece de forma superexponencial.
Veamos cómo se comportan BFS y A* con 4 y 5 bloques.

### Problema de 4 bloques

Estado inicial: A, B, C, D todos en la mesa.
Meta: torre A/B/C/D (A arriba, D abajo).

In [10]:
# ── 4-block problem ───────────────────────────────────────────────────────────
blocks_4 = ["A", "B", "C", "D"]
actions_4 = make_blocks_world_actions(blocks_4)

initial_4 = frozenset(
    {f"On({b},Mesa)" for b in blocks_4} | {f"Clear({b})" for b in blocks_4}
)

goal_4 = frozenset({
    "On(A,B)", "On(B,C)", "On(C,D)", "On(D,Mesa)", "Clear(A)"
})

print(f"Bloques: {blocks_4}")
print(f"Acciones totales: {len(actions_4)}")
print(f"Estado inicial: todos en la mesa")
print(f"Meta: torre A/B/C/D")
print()

# ── BFS ────────────────────────────────────────────────────────────────────────
r_bfs_4 = benchmark("BFS", forward_bfs, initial_4, goal_4, actions_4)
print(f"BFS: plan={r_bfs_4['plan_length']} pasos, nodos={r_bfs_4['nodes_explored']}, "
      f"tiempo={r_bfs_4['time_ms']:.1f} ms")

# ── A* ─────────────────────────────────────────────────────────────────────────
r_astar_4 = benchmark("A*", forward_astar, initial_4, goal_4, actions_4)
print(f"A*:  plan={r_astar_4['plan_length']} pasos, nodos={r_astar_4['nodes_explored']}, "
      f"tiempo={r_astar_4['time_ms']:.1f} ms")

print(f"\nA* exploró {r_bfs_4['nodes_explored'] - r_astar_4['nodes_explored']} nodos menos que BFS.")

# Show plan
print("\nPlan A*:")
for i, a in enumerate(r_astar_4['plan'], 1):
    print(f"  Paso {i}: {a.name}")

Bloques: ['A', 'B', 'C', 'D']
Acciones totales: 48
Estado inicial: todos en la mesa
Meta: torre A/B/C/D

BFS: plan=3 pasos, nodos=67, tiempo=1.0 ms
A*:  plan=3 pasos, nodos=4, tiempo=10.6 ms

A* exploró 63 nodos menos que BFS.

Plan A*:
  Paso 1: MoverDesdeMesa(C,D)
  Paso 2: MoverDesdeMesa(B,C)
  Paso 3: MoverDesdeMesa(A,B)


### Problema de 5 bloques

Estado inicial: A, B, C, D, E todos en la mesa.
Meta: torre A/B/C/D/E.

Aqui el espacio de estados es mucho más grande. BFS puede tardar significativamente
más que A*. Observa cómo crece el número de nodos explorados.

In [11]:
# ── 5-block problem ───────────────────────────────────────────────────────────
blocks_5 = ["A", "B", "C", "D", "E"]
actions_5 = make_blocks_world_actions(blocks_5)

initial_5 = frozenset(
    {f"On({b},Mesa)" for b in blocks_5} | {f"Clear({b})" for b in blocks_5}
)

goal_5 = frozenset({
    "On(A,B)", "On(B,C)", "On(C,D)", "On(D,E)", "On(E,Mesa)", "Clear(A)"
})

print(f"Bloques: {blocks_5}")
print(f"Acciones totales: {len(actions_5)}")
print(f"Estado inicial: todos en la mesa")
print(f"Meta: torre A/B/C/D/E")
print()

# ── BFS ────────────────────────────────────────────────────────────────────────
print("Ejecutando BFS... (puede tardar unos segundos)")
r_bfs_5 = benchmark("BFS", forward_bfs, initial_5, goal_5, actions_5)
print(f"BFS: plan={r_bfs_5['plan_length']} pasos, nodos={r_bfs_5['nodes_explored']}, "
      f"tiempo={r_bfs_5['time_ms']:.1f} ms")

# ── A* ─────────────────────────────────────────────────────────────────────────
print("\nEjecutando A*...")
r_astar_5 = benchmark("A*", forward_astar, initial_5, goal_5, actions_5)
print(f"A*:  plan={r_astar_5['plan_length']} pasos, nodos={r_astar_5['nodes_explored']}, "
      f"tiempo={r_astar_5['time_ms']:.1f} ms")

if r_bfs_5['nodes_explored'] > r_astar_5['nodes_explored']:
    ratio = r_bfs_5['nodes_explored'] / r_astar_5['nodes_explored']
    print(f"\nA* exploró {ratio:.1f}x menos nodos que BFS.")

# Show A* plan
print("\nPlan A*:")
for i, a in enumerate(r_astar_5['plan'], 1):
    print(f"  Paso {i}: {a.name}")

Bloques: ['A', 'B', 'C', 'D', 'E']
Acciones totales: 100
Estado inicial: todos en la mesa
Meta: torre A/B/C/D/E

Ejecutando BFS... (puede tardar unos segundos)
BFS: plan=4 pasos, nodos=477, tiempo=25.3 ms

Ejecutando A*...
A*:  plan=4 pasos, nodos=5, tiempo=643.7 ms

A* exploró 95.4x menos nodos que BFS.

Plan A*:
  Paso 1: MoverDesdeMesa(D,E)
  Paso 2: MoverDesdeMesa(C,D)
  Paso 3: MoverDesdeMesa(B,C)
  Paso 4: MoverDesdeMesa(A,B)


### Tabla resumen: crecimiento de la complejidad

Comparamos los resultados para 3, 4 y 5 bloques. El crecimiento del espacio de estados
es evidente: más bloques implican exponencialmente más nodos que explorar. La heurística
de delete relaxation ayuda a A* a podar grandes porciones del espacio.

In [12]:
print("=" * 70)
print("TABLA RESUMEN: Crecimiento de complejidad en Blocks World")
print("=" * 70)
print(f"{'Bloques':>8} {'Acciones':>10} {'Algo':>6} {'Plan':>6} {'Nodos':>8} {'Tiempo (ms)':>12}")
print("-" * 70)

all_results = [
    (3, len(actions_3), r_bfs_3, r_astar_3),
    (4, len(actions_4), r_bfs_4, r_astar_4),
    (5, len(actions_5), r_bfs_5, r_astar_5),
]

for n_blocks, n_actions, r_bfs, r_astar in all_results:
    for r in [r_bfs, r_astar]:
        plan_str = str(r['plan_length']) if r['plan_length'] else "N/A"
        print(f"{n_blocks:>8} {n_actions:>10} {r['name']:>6} {plan_str:>6} "
              f"{r['nodes_explored']:>8} {r['time_ms']:>12.2f}")
    print()  # blank line between block sizes

print("Observaciones:")
print("  - El número de acciones crece como O(n^3)")
print("  - BFS explora muchos más nodos conforme crece n")
print("  - A* con delete-relaxation poda significativamente el espacio")
print("  - Ambos encuentran planes óptimos (misma longitud)")

TABLA RESUMEN: Crecimiento de complejidad en Blocks World
 Bloques   Acciones   Algo   Plan    Nodos  Tiempo (ms)
----------------------------------------------------------------------
       3         18    BFS      2       11         0.26
       3         18     A*      2        3         0.56

       4         48    BFS      3       67         1.00
       4         48     A*      3        4        10.62

       5        100    BFS      4      477        25.26
       5        100     A*      4        5       643.66

Observaciones:
  - El número de acciones crece como O(n^3)
  - BFS explora muchos más nodos conforme crece n
  - A* con delete-relaxation poda significativamente el espacio
  - Ambos encuentran planes óptimos (misma longitud)


---
## Sección 7: Búsqueda hacia atrás (regresión)

La búsqueda hacia adelante parte de $s_0$ y aplica acciones hasta alcanzar $G$.
La búsqueda **hacia atrás** hace lo contrario: parte de la meta $G$ y pregunta
"¿qué acción pudo haber producido esta situación?", retrocediendo hasta $s_0$.

Conceptos clave:

- **Subobjetivo**: conjunto parcial de proposiciones que necesitamos (no un estado completo).
- **Acción relevante**: $\text{Add}(a) \cap g \neq \emptyset$ — produce algo que necesitamos.
- **Acción consistente**: $\text{Del}(a) \cap g = \emptyset$ — no destruye nada que necesitamos.
- **Regresión**: $\text{regress}(g, a) = (g - \text{Add}(a)) \cup \text{Pre}(a)$

La búsqueda termina cuando el subobjetivo es un subconjunto del estado inicial: $g \subseteq s_0$.

In [13]:
def get_relevant_consistent(subgoal, all_actions):
    """
    Find all actions that are relevant AND consistent with a subgoal.

    Relevant:   Add(a) ∩ g ≠ ∅  — the action achieves at least one needed proposition.
    Consistent: Del(a) ∩ g = ∅  — the action doesn't destroy any needed proposition.

    Args:
        subgoal     : frozenset — set of propositions we need to be true
        all_actions : list[Action] — all ground actions

    Returns:
        list[Action] — actions passing both filters
    """
    result = []
    for a in all_actions:
        # Relevant: does the action produce something we need?
        if a.add_list & subgoal:             # Add(a) ∩ g ≠ ∅
            # Consistent: does the action NOT destroy something we need?
            if not (a.delete_list & subgoal): # Del(a) ∩ g = ∅
                result.append(a)
    return result


def regress(subgoal, action):
    """
    Compute the regression of a subgoal through an action.

    Formula: regress(g, a) = (g - Add(a)) ∪ Pre(a)

    Step 1: Remove from g the propositions that action a will produce.
    Step 2: Add the preconditions of a (we now need these to be true before a).

    Args:
        subgoal : frozenset — current subgoal
        action  : Action   — relevant and consistent action

    Returns:
        frozenset — new subgoal (what must be true before executing a)
    """
    return (subgoal - action.add_list) | action.preconditions


def backward_bfs(initial, goal, all_actions, verbose=False):
    """
    Backward planning with BFS (FIFO frontier).

    Starts from the goal G and regresses until reaching a subgoal
    that is a subset of the initial state s0.

    Args:
        initial     : frozenset — initial state s0
        goal        : frozenset — goal propositions G
        all_actions : list[Action] — all ground actions
        verbose     : bool — if True, print detailed trace

    Returns:
        tuple(list[Action], int) — (plan, nodes_explored)
        or (None, nodes_explored) if no plan exists
    """
    # ── Initialization ────────────────────────────────────────────────────
    frontier = deque([goal])             # [B1] start from GOAL, not s0
    explored = set()
    parent = {goal: (None, None)}        # subgoal -> (parent_subgoal, action)
    nodes_explored = 0

    # ── Main loop ─────────────────────────────────────────────────────────
    while frontier:
        g = frontier.popleft()           # g is a SUBGOAL, not a complete state
        nodes_explored += 1

        if verbose:
            print(f"\n--- Iteración {nodes_explored} ---")
            print(f"  Subobjetivo: {format_state(g)}")

        # ── Termination test [B2] ─────────────────────────────────────────
        if g.issubset(initial):          # [B2] is the initial state enough?
            if verbose:
                print(f"  ** g ⊆ s0 — PLAN ENCONTRADO **")
            # Reconstruct plan — walk parent chain from g (near s0) to G.
            # parent[g_i] = (g_{i-1}, a_i) means: executing a_i in a state
            # satisfying g_i produces a state satisfying g_{i-1}.
            # Walking from g_final toward G collects actions in execution
            # order (from s0 toward G), so NO reverse is needed.
            plan = []
            current = g
            while parent[current][0] is not None:
                prev_subgoal, action = parent[current]
                plan.append(action)
                current = prev_subgoal
            return plan, nodes_explored

        explored.add(g)

        # ── Expand: relevant + consistent actions [B3] ────────────────────
        relevant = get_relevant_consistent(g, all_actions)
        if verbose:
            print(f"  Acciones relevantes y consistentes: {len(relevant)}")

        new_count = 0
        for a in relevant:
            new_g = regress(g, a)        # [B3] regression
            if new_g not in explored and new_g not in set(frontier):
                parent[new_g] = (g, a)
                frontier.append(new_g)
                new_count += 1
                if verbose:
                    print(f"    + {a.name} -> {format_state(new_g)}")

        if verbose:
            print(f"  Nuevos subobjetivos: {new_count}")
            print(f"  Tamaño de frontera: {len(frontier)}")

    return None, nodes_explored


print("backward_bfs definida — funciones: get_relevant_consistent, regress, backward_bfs")

backward_bfs definida — funciones: get_relevant_consistent, regress, backward_bfs


### Resolver el problema de 3 bloques con búsqueda hacia atrás

Mismo problema: A, B, C en la mesa → torre A/B/C.
Ejecutamos con `verbose=True` para ver la regresión paso a paso.

In [14]:
# ── Backward BFS on 3-block problem ──────────────────────────────────────
print("Estado inicial:", format_state(initial_3))
print("Meta:          ", format_state(goal_3))
print()

plan_back_3, explored_back_3 = backward_bfs(initial_3, goal_3, actions_3, verbose=True)

print("\n" + "=" * 60)
print("RESUMEN — Backward BFS Planning (3 bloques)")
print("=" * 60)
print(f"Plan encontrado: {len(plan_back_3)} pasos")
print(f"Nodos explorados (subobjetivos): {explored_back_3}")
print()
print("Plan (orden de ejecución):")
for i, action in enumerate(plan_back_3, 1):
    print(f"  Paso {i}: {action.name}")

# Verify: execute the plan forward
print("\nVerificación (ejecutando el plan hacia adelante):")
state = initial_3
for i, action in enumerate(plan_back_3, 1):
    state = apply_action(state, action)
    print(f"  Paso {i}: {action.name} -> {format_state(state)}")
print(f"\nMeta alcanzada: {goal_3.issubset(state)}")

Estado inicial: {Clear(A), Clear(B), Clear(C), On(A,Mesa), On(B,Mesa), On(C,Mesa)}
Meta:           {Clear(A), On(A,B), On(B,C), On(C,Mesa)}


--- Iteración 1 ---
  Subobjetivo: {Clear(A), On(A,B), On(B,C), On(C,Mesa)}
  Acciones relevantes y consistentes: 8
    + Mover(A,C,B) -> {Clear(A), Clear(B), On(A,C), On(B,C), On(C,Mesa)}
    + MoverDesdeMesa(A,B) -> {Clear(A), Clear(B), On(A,Mesa), On(B,C), On(C,Mesa)}
    + Mover(B,A,C) -> {Clear(B), Clear(C), On(A,B), On(B,A), On(C,Mesa)}
    + MoverAMesa(B,A) -> {Clear(B), On(A,B), On(B,A), On(B,C), On(C,Mesa)}
    + MoverDesdeMesa(B,C) -> {Clear(A), Clear(B), Clear(C), On(A,B), On(B,Mesa), On(C,Mesa)}
    + Mover(C,A,B) -> {Clear(B), Clear(C), On(A,B), On(B,C), On(C,A), On(C,Mesa)}
    + MoverAMesa(C,A) -> {Clear(C), On(A,B), On(B,C), On(C,A)}
    + MoverAMesa(C,B) -> {Clear(A), Clear(C), On(A,B), On(B,C), On(C,B)}
  Nuevos subobjetivos: 8
  Tamaño de frontera: 8

--- Iteración 2 ---
  Subobjetivo: {Clear(A), Clear(B), On(A,C), On(B,C), On(

### Comparación: Forward vs Backward

Comparamos ambas direcciones de búsqueda en el problema de 3 bloques.
Observa que ambos encuentran el mismo plan óptimo, pero pueden explorar
diferente cantidad de nodos.

In [15]:
# ── Compare forward BFS vs backward BFS ──────────────────────────────────
print("=" * 65)
print("COMPARACIÓN: Forward BFS vs Backward BFS (3 bloques)")
print("=" * 65)
print(f"{'Dirección':<20} {'Plan':>6} {'Nodos explorados':>18}")
print("-" * 50)

# Forward BFS (already computed)
print(f"{'Forward BFS':<20} {len(plan_3):>6} {explored_3:>18}")

# Backward BFS
print(f"{'Backward BFS':<20} {len(plan_back_3):>6} {explored_back_3:>18}")

print()
print("Plan forward: ", " → ".join(a.name for a in plan_3))
print("Plan backward:", " → ".join(a.name for a in plan_back_3))

if [a.name for a in plan_3] == [a.name for a in plan_back_3]:
    print("\nMismo plan encontrado por ambas direcciones.")
else:
    print("\nPlanes diferentes (misma longitud óptima).")

COMPARACIÓN: Forward BFS vs Backward BFS (3 bloques)
Dirección              Plan   Nodos explorados
--------------------------------------------------
Forward BFS               2                 11
Backward BFS              2                 21

Plan forward:  MoverDesdeMesa(B,C) → MoverDesdeMesa(A,B)
Plan backward: MoverDesdeMesa(B,C) → MoverDesdeMesa(A,B)

Mismo plan encontrado por ambas direcciones.
